In [67]:
import io
import functions as DS
from pptx import Presentation
import pandas as pd

### How to Build a PowerPoint Template

This guide explains how to create PowerPoint templates optimized for automation. All elements should be placed in normal slide layouts, not master slides, unless specified otherwise. include the bg color in the master

### Tables
- Include tables directly in your template slides
- Name cell A1 (first cell) with format: `table name: {Unique ID}`
- All formatting properties of the table will be preserved
- Example: `table name: quarterly_results`

### Images
- **Static images** (logos, watermarks) should be placed in the master slide
- **Dynamic images** require a placeholder shape:
- Insert a shape where you want the image to appear
- Name the shape: `image placeholder: {Unique ID}`
- Images will be inserted from the top-right corner of this placeholder
- Example: `image placeholder: Creative`

### Text
- For dynamic text content:
- Create a text box where text should appear
- Name the text box: `PLACE_TEXT_{UniqueID}`
- All formatting properties will be preserved
- Example: `PLACE_TEXT_header_title`

### Charts...
- rn just do images but we need to figure this out

### Best Practices
- Use descriptive Unique IDs that reflect content purpose
- Maintain consistent naming conventions across slides
- Test your template with sample data before full implementation
- Document all placeholder IDs for reference when writing automation code


# Once template is set up, build navigation_screen

## Set up env!

In [68]:
pptx_file = "template.pptx"  # Replace with your PPTX file path
prs = Presentation(pptx_file)
navigation_screen = DS.create_navigation_screen(prs)


--- Template contains... ---
titleslide - in position of 0
subtitleslide - in position of 1
smsdataslide - in position of 2


# ADD TITLE SLIDE

In [69]:
prs = DS.duplicate_slide(prs, navigation_screen['titleslide'],verbose=False)
new_slide = prs.slides[-1]

#add title slide
DS.find_replace_text(new_slide,'PLACE_TEXT_Title','SMS Engagement',verbose=True)


Original Text: place_text_title
New Text: SMS Engagement



# Add Data TITLE - SMS

In [71]:
#add data slide
prs = DS.duplicate_slide(prs, navigation_screen['subtitleslide'],verbose=True)
new_slide = prs.slides[-1]

#replace title
DS.find_replace_text(new_slide,'PLACE_TEXT_TITLE','Touch Reviews',verbose=True)



--- Text in slide ---
PLACE_TEXT_TITLE

--- Tables in slide ---
--- Images in slide ---
Original Text: place_text_title

New Text: Touch Reviews



# CLEAN SMS DATA


In [57]:
file = "Apple NPI 25 Clean Data.xlsx"
data = pd.read_excel(file, sheet_name="SMS")

In [58]:
def combine_fields(row):
    parts = [row['Cohort']]
    if pd.notna(row['Audience Details 1']):
        parts.append(row['Audience Details 1'])
    if pd.notna(row['Audience Details 2']):
        parts.append(row['Audience Details 2'])
    return '_'.join(parts)

def format_number(x):
    if x >= 1_000_000:
        return f"{x/1_000_000:.2f}M"
    elif x >= 1_000:
        return f"{x/1_000:.1f}K"
    else:
        return str(x)
def format_perc(x):
    return f"{x*100:.2f}%"


In [59]:
data['Touch'] = data['Touch'].str.title()
data['Cohort'] = data['Cohort'].str.title()
data['Cohort'] = data['Cohort'].fillna('Missing_TOUCH')

data['Audience'] = data.apply(combine_fields, axis=1)

data.rename(columns={
    "Deliveries": "Volume",
    "Branch CTR": "CTR"
}, inplace=True)

data = data[['Touch','Audience',
    'Creative',
    'Volume',
    'CTR',
]]



# Apply to your DataFrame
data['Volume'] = data['Volume'].apply(format_number)
data['CTR'] = data['CTR'].apply(format_perc)

# 

In [60]:
data

,Touch,Audience,Creative,Volume,CTR
0,Preorder,Churn_UPGRADE,T-Mobile: You’re eligible for iPhone 17 Pro On...,50.7K,8.56%
1,Preorder,Growth_AAL,T-Mobile: You’re eligible for up to $830 off i...,2.47M,3.53%
2,Announce,Missing_TOUCH,T-Mobile: Countdown’s on. Preorder the phone y...,8.16M,6.87%
3,Launch,Churn_UPGRADE_DCC,T-Mobile: You’re eligible for iPhone 17 Pro On...,34.1K,7.47%
4,Launch,Churn_UPGRADE,T-Mobile: You’re eligible for iPhone 17 Pro On...,20.7K,4.90%
5,Launch,Growth_AAL_DCC,T-Mobile: You're eligible for up to $830 off i...,1.21M,1.87%
6,Launch,Growth_AAL,T-Mobile: You’re eligible for up to $830 off i...,1.25M,1.74%


# Add Data SMS

In [61]:
row_count = 4 #break up data into chunks of X
chunks = [data.iloc[i:i+row_count] for i in range(0, len(data), row_count)]


In [62]:
for chunk in chunks:
    display(chunk)
    prs = DS.duplicate_slide(prs, navigation_screen['smsdataslide'],verbose=False)
    new_slide = prs.slides[-1]

    title_name = " & ".join(chunk['Touch'].dropna().unique().astype(str))
    print(title_name)
    DS.find_replace_text(new_slide,'PLACE_TEXT_TITLE',title_name,verbose=False)


    DS.add_data_table_new(new_slide,'Touch',chunk)


,Touch,Audience,Creative,Volume,CTR
0,Preorder,Churn_UPGRADE,T-Mobile: You’re eligible for iPhone 17 Pro On...,50.7K,8.56%
1,Preorder,Growth_AAL,T-Mobile: You’re eligible for up to $830 off i...,2.47M,3.53%
2,Announce,Missing_TOUCH,T-Mobile: Countdown’s on. Preorder the phone y...,8.16M,6.87%
3,Launch,Churn_UPGRADE_DCC,T-Mobile: You’re eligible for iPhone 17 Pro On...,34.1K,7.47%


Preorder & Announce & Launch
Data fits table Touch, updating cells with formatting...


,Touch,Audience,Creative,Volume,CTR
4,Launch,Churn_UPGRADE,T-Mobile: You’re eligible for iPhone 17 Pro On...,20.7K,4.90%
5,Launch,Growth_AAL_DCC,T-Mobile: You're eligible for up to $830 off i...,1.21M,1.87%
6,Launch,Growth_AAL,T-Mobile: You’re eligible for up to $830 off i...,1.25M,1.74%


Launch
Data fits table Touch, updating cells with formatting...


# Save File

In [63]:
indices_to_delete = list(navigation_screen.values())
    
    # Sort in reverse order to avoid shifting issues
for i in sorted(indices_to_delete, reverse=True):
    rId = prs.slides._sldIdLst[i]
    prs.slides._sldIdLst.remove(rId)
    print(f"Deleted slide at index {i}")

Deleted slide at index 2
Deleted slide at index 1
Deleted slide at index 0


In [64]:
prs.save('output.pptx')